In [2]:
!pip -q install transformers peft accelerate

import os, math, random, copy
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from google.colab import drive

drive.mount("/content/drive")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

ROOT = "/content/drive/My Drive/Lora Exp"
MODEL_PATH = f"{ROOT}/Models/gpt-neo-125m"
DATA1_DIR  = f"{ROOT}/Data1"
DATA_DIR   = f"{ROOT}/Data"
SAVE_ROOT  = f"{ROOT}/Unlearned_models"
os.makedirs(DATA1_DIR, exist_ok=True)
os.makedirs(SAVE_ROOT, exist_ok=True)

# Choose retain corpus file (will be truncated to a fixed token budget below)
RETAIN_PATH = f"{DATA_DIR}/roy.txt"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



Mounted at /content/drive
device: cpu


## **Tokenizer and Data**

In [ ]:
PREFIX_PATH = f"{DATA1_DIR}/train_prefix.npy"
SUFFIX_PATH = f"{DATA1_DIR}/train_suffix.npy"

!wget -O "{PREFIX_PATH}" "https://github.com/ethz-spylab/lm-extraction-benchmark-data/raw/main/datasets/train_prefix.npy"
!wget -O "{SUFFIX_PATH}" "https://github.com/ethz-spylab/lm-extraction-benchmark-data/raw/main/datasets/train_suffix.npy"


--2026-02-19 00:53:39--  https://github.com/ethz-spylab/lm-extraction-benchmark-data/raw/main/datasets/train_prefix.npy
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ethz-spylab/lm-extraction-benchmark-data/main/datasets/train_prefix.npy [following]
--2026-02-19 00:53:39--  https://raw.githubusercontent.com/ethz-spylab/lm-extraction-benchmark-data/main/datasets/train_prefix.npy
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1500128 (1.4M) [application/octet-stream]
Saving to: ‘/content/drive/My Drive/Lora Exp/Data1/train_prefix.npy’

/content/drive/My D 100%[===================>]   1.43M  --.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(device)
base_model.eval()
print("Loaded base model + tokenizer")


Loading weights:   0%|          | 0/160 [00:02<?, ?it/s]

Loaded base model + tokenizer


In [ ]:
prefixes = np.load(PREFIX_PATH, allow_pickle=True)  # (15000, 50)
suffixes = np.load(SUFFIX_PATH, allow_pickle=True)  # (15000, 50)
assert prefixes.shape == (15000, 50) and suffixes.shape == (15000, 50)

NUM_SEQS = 200
TOKENS_PER_SEQ = 200

idx = list(range(len(prefixes)))
random.Random(SEED).shuffle(idx)

forget_lines = []
forget_ids_200 = []  # keep token-exact IDs for eval if you want

for k in range(NUM_SEQS):
    i1 = idx[2*k]
    i2 = idx[2*k + 1]

    ids200 = (
        list(map(int, prefixes[i1])) + list(map(int, suffixes[i1])) +
        list(map(int, prefixes[i2])) + list(map(int, suffixes[i2]))
    )
    assert len(ids200) == TOKENS_PER_SEQ
    forget_ids_200.append(ids200)

    txt = tokenizer.decode(ids200).replace("\n", " ").replace("\r", " ").strip()
    forget_lines.append(txt)

FORGET_PATH = f"{DATA1_DIR}/tdec_forget_{NUM_SEQS}x{TOKENS_PER_SEQ}.txt"
with open(FORGET_PATH, "w", encoding="utf-8") as f:
    for line in forget_lines:
        f.write(line + "\n")

print("Saved forget:", FORGET_PATH, "lines:", len(forget_lines))


Saved forget: /content/drive/My Drive/Lora Exp/Data1/tdec_forget_200x200.txt lines: 200


In [ ]:
def read_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

retain_full_text = read_text(RETAIN_PATH)

RETAIN_EVAL_TOKENS = 50_000
retain_ids_full = tokenizer(retain_full_text, return_tensors="pt", truncation=False).input_ids[0]
retain_ids_eval = retain_ids_full[:RETAIN_EVAL_TOKENS].to(device)
retain_eval_text = tokenizer.decode(retain_ids_eval)

RETAIN_50K_PATH = f"{DATA1_DIR}/retain_{RETAIN_EVAL_TOKENS}_tokens.txt"
with open(RETAIN_50K_PATH, "w", encoding="utf-8") as f:
    f.write(retain_eval_text)

print("Saved retain 50k:", RETAIN_50K_PATH, "tokens:", retain_ids_eval.numel())


Token indices sequence length is longer than the specified maximum sequence length for this model (151128 > 2048). Running this sequence through the model will result in indexing errors


Saved retain 50k: /content/drive/My Drive/Lora Exp/Data1/retain_50000_tokens.txt tokens: 50000


## **PPL Function**

In [ ]:
PPL_MAX_LEN = 512
PPL_STRIDE  = 256

def ppl(model, input_ids_1d, max_len=PPL_MAX_LEN, stride=PPL_STRIDE):
    """
    Strided sliding-window perplexity for fixed-length causal LMs.
    Keep max_len/stride fixed across ALL baselines and experiments. [web:32]
    """
    model.eval()
    seq_len = input_ids_1d.size(0)
    total_nll, total_pred = 0.0, 0

    with torch.no_grad():
        for start in range(0, seq_len - 1, stride):
            end = min(start + max_len, seq_len)
            chunk = input_ids_1d[start:end].unsqueeze(0).to(device)
            if chunk.size(1) < 2:
                break
            out = model(input_ids=chunk, labels=chunk)
            pred = chunk.size(1) - 1
            total_nll += out.loss.item() * pred
            total_pred += pred
            if end == seq_len:
                break

    return math.exp(total_nll / total_pred)


## **Baseline PPL**

In [ ]:
forget_eval_ids = tokenizer("\n".join(forget_lines), return_tensors="pt", truncation=False).input_ids[0].to(device)
retain_eval_ids = tokenizer(retain_eval_text, return_tensors="pt", truncation=False).input_ids[0].to(device)

print("forget eval tokens:", forget_eval_ids.numel())
print("retain eval tokens:", retain_eval_ids.numel())

forget eval tokens: 38951
retain eval tokens: 50000


In [ ]:


BASE_FORGET = ppl(base_model, forget_eval_ids)
BASE_RETAIN = ppl(base_model, retain_eval_ids)

print("BASE_FORGET:", BASE_FORGET)
print("BASE_RETAIN:", BASE_RETAIN)


BASE_FORGET: 11.77557685978259
BASE_RETAIN: 38.02937180100788


## **Training Batches and LoRA for unlearning**

In [ ]:
TRAIN_SEQ_LEN = 200
BATCH_SIZE = 4

def tokenize_fixed(lines, max_len=TRAIN_SEQ_LEN):
    enc = tokenizer(lines, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]

# Forget training set
forget_train_ids, forget_train_mask = tokenize_fixed(forget_lines)

# Retain training set from the SAME retain_eval_text (not full Byron!)
retain_ids_full_train = tokenizer(retain_eval_text, return_tensors="pt", truncation=False).input_ids[0]
retain_chunks = []
for i in range(0, len(retain_ids_full_train), TRAIN_SEQ_LEN):
    chunk = retain_ids_full_train[i:i+TRAIN_SEQ_LEN]
    if chunk.numel() < 50:
        continue
    if chunk.numel() < TRAIN_SEQ_LEN:
        chunk = F.pad(chunk, (0, TRAIN_SEQ_LEN - chunk.numel()), value=tokenizer.pad_token_id)
    retain_chunks.append(chunk)

retain_train_ids = torch.stack(retain_chunks)
retain_train_mask = (retain_train_ids != tokenizer.pad_token_id).long()

print("forget train:", forget_train_ids.shape)
print("retain train:", retain_train_ids.shape)


forget train: torch.Size([200, 200])
retain train: torch.Size([250, 200])


In [ ]:
class Iterator:
    def __init__(self, ids, mask, bs=BATCH_SIZE, seed=SEED):
        self.ids = ids
        self.mask = mask
        self.bs = bs
        self.gen = torch.Generator().manual_seed(seed)
        self.idx = torch.randperm(len(ids), generator=self.gen)
        self.pos = 0

    def batch(self):
        if self.pos + self.bs > len(self.idx):
            self.idx = torch.randperm(len(self.ids), generator=self.gen)
            self.pos = 0
        b = self.idx[self.pos:self.pos+self.bs]
        self.pos += self.bs
        return self.ids[b].to(device), self.mask[b].to(device)


## **Defining Unlearning Methods, Loss and LoRA**

In [ ]:
def make_lora(rank, alpha=None, dropout=0.05):
    if alpha is None:
        alpha = 2 * rank
    m = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(device)
    cfg = LoraConfig(
        r=rank, lora_alpha=alpha, lora_dropout=dropout,
        target_modules=["q_proj", "k_proj", "v_proj"],
        bias="none", task_type="CAUSAL_LM"
    )
    m = get_peft_model(m, cfg)
    m.print_trainable_parameters()
    return m

def ce_loss(model, input_ids, attn_mask):
    labels = input_ids.clone()
    labels[attn_mask == 0] = -100
    return model(input_ids=input_ids, attention_mask=attn_mask, labels=labels).loss

def run_GD(model, forget_iter, retain_iter, lr=2e-5, steps=200, lambda_retain=4.0, clip=1.0):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for step in range(steps):
        x_f, m_f = forget_iter.batch()
        x_r, m_r = retain_iter.batch()
        loss_f = ce_loss(model, x_f, m_f)
        loss_r = ce_loss(model, x_r, m_r)
        total = -loss_f + lambda_retain * loss_r
        opt.zero_grad()
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        if step % 100 == 0:
            print(f"[GD] {step}/{steps} forget={loss_f.item():.4f} retain={loss_r.item():.4f}")
    model.eval()


In [ ]:
# ===============================
class SineLoRALinear(nn.Module):
    def __init__(self, base_layer, r, alpha, omega):
        super().__init__()

        self.base = base_layer
        self.r = r
        self.alpha = alpha
        self.omega = omega
        self.scaling = alpha / r  # Standard LoRA scaling

        in_dim = base_layer.in_features
        out_dim = base_layer.out_features
        device = base_layer.weight.device

        # ✅ PAPER EXACT: A=[in_dim, r], B=[r, out_dim] (not Linear layers)
        self.A = nn.Parameter(torch.empty(in_dim, r, device=device, dtype=torch.float32))
        self.B = nn.Parameter(torch.empty(r, out_dim, device=device, dtype=torch.float32))

        # ✅ PAPER-RECOMMENDED Initialization
        nn.init.normal_(self.A, std=0.02)
        nn.init.normal_(self.B, std=0.02)  # Non-zero B init (your zero-init was slow)

        # Freeze base layer
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        base_out = self.base(x)

        # ✅ PAPER EXACT: x @ sin(ω × (Aᵀ @ Bᵀ))
        low_rank_weight = self.A @ self.B  # [in_dim, out_dim]
        sine_weight = torch.sin(self.omega * low_rank_weight)  # Elementwise sine

        # Final low-rank delta
        lora_out = (x @ sine_weight) * self.scaling
        return base_out + lora_out


def make_sine_lora(rank, omega=1.5, alpha=None):
    if alpha is None: alpha = 2 * rank
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(device)

    # 1. Apply Standard LoRA to Attention (via PEFT)
    from peft import LoraConfig, get_peft_model
    peft_config = LoraConfig(
        r=rank, lora_alpha=alpha,
        target_modules=["q_proj", "v_proj", "k_proj"], # Attention only
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, peft_config)

    # 2. Manually wrap MLP layers with your SineLoRALinear
    target_mlp = ["c_fc", "c_proj"]
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and any(name.endswith(t) for t in target_mlp):
            parent_name = name.rsplit(".", 1)[0]
            child_name = name.split(".")[-1]
            parent = dict(model.named_modules())[parent_name]

            # Replace with Sine-LoRA
            new_layer = SineLoRALinear(module, rank, alpha, omega)

            # STABILITY FIX: Initialize B to zero so we start at base model state
            nn.init.zeros_(new_layer.B)
            setattr(parent, child_name, new_layer)

    return model

In [ ]:
SAVE_ROOT  = f"{ROOT}/Unlearned_models/LoraGDnLayers"

## **Unlearning EXP**

# GD on Sine Lora


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoModelForCausalLM

RANKS = [2, 4, 8, 16]
OMEGA = 12

for RANK in RANKS:
    print(f"\n===== GD | Sine-LoRA | rank={RANK} | omega={OMEGA} =====")

    model = make_sine_lora(RANK, omega=OMEGA)

    forget_iter = Iterator(
        forget_train_ids,
        forget_train_mask,
        seed=SEED
    )
    retain_iter = Iterator(
        retain_train_ids,
        retain_train_mask,
        seed=SEED + 1
    )

    run_GD(
        model,
        forget_iter,
        retain_iter,
        lr=1e-5,
        steps=150,
        lambda_retain=5.0,
        clip=0.3
    )

    merged = model
    pF = ppl(merged, forget_eval_ids)
    pR = ppl(merged, retain_eval_ids)

    print(
        f"GD(Sine) rank={RANK} | "
        f"Forget: {BASE_FORGET:.2f} → {pF:.2f} | "
        f"Retain: {BASE_RETAIN:.2f} → {pR:.2f}"
    )

    save_dir = (
        f"{SAVE_ROOT}/GD_Sine_rank{RANK}"
        f"_omega{OMEGA}_seed{SEED}"
    )
    merged.save_pretrained(save_dir)
    print("saved:", save_dir)



===== GD | Sine-LoRA | rank=2 | omega=12 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2604 retain=3.7700
GD(Sine) rank=2 | Forget: 11.78 → 11.98 | Retain: 38.03 → 36.51
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank2_omega12_seed42

===== GD | Sine-LoRA | rank=4 | omega=12 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2707 retain=3.7494
GD(Sine) rank=4 | Forget: 11.78 → 12.19 | Retain: 38.03 → 35.75
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank4_omega12_seed42

===== GD | Sine-LoRA | rank=8 | omega=12 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.3019 retain=3.7086
GD(Sine) rank=8 | Forget: 11.78 → 13.04 | Retain: 38.03 → 34.54
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank8_omega12_seed42

===== GD | Sine-LoRA | rank=16 | omega=12 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.3824 retain=3.6579
GD(Sine) rank=16 | Forget: 11.78 → 16.10 | Retain: 38.03 → 33.24
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank16_omega12_seed42


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoModelForCausalLM

RANKS = [2, 4, 8, 16]
OMEGA = 15

for RANK in RANKS:
    print(f"\n===== GD | Sine-LoRA | rank={RANK} | omega={OMEGA} =====")

    model = make_sine_lora(RANK, omega=OMEGA)

    forget_iter = Iterator(
        forget_train_ids,
        forget_train_mask,
        seed=SEED
    )
    retain_iter = Iterator(
        retain_train_ids,
        retain_train_mask,
        seed=SEED + 1
    )

    run_GD(
        model,
        forget_iter,
        retain_iter,
        lr=1e-5,
        steps=150,
        lambda_retain=5.0,
        clip=0.3
    )

    merged = model
    pF = ppl(merged, forget_eval_ids)
    pR = ppl(merged, retain_eval_ids)

    print(
        f"GD(Sine) rank={RANK} | "
        f"Forget: {BASE_FORGET:.2f} → {pF:.2f} | "
        f"Retain: {BASE_RETAIN:.2f} → {pR:.2f}"
    )

    save_dir = (
        f"{SAVE_ROOT}/GD_Sine_rank{RANK}"
        f"_omega{OMEGA}_seed{SEED}"
    )
    merged.save_pretrained(save_dir)
    print("saved:", save_dir)



===== GD | Sine-LoRA | rank=2 | omega=15 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2615 retain=3.7657
GD(Sine) rank=2 | Forget: 11.78 → 12.00 | Retain: 38.03 → 36.35
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank2_omega15_seed42

===== GD | Sine-LoRA | rank=4 | omega=15 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2790 retain=3.7349
GD(Sine) rank=4 | Forget: 11.78 → 12.38 | Retain: 38.03 → 35.35
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank4_omega15_seed42

===== GD | Sine-LoRA | rank=8 | omega=15 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.3210 retain=3.6947
GD(Sine) rank=8 | Forget: 11.78 → 13.38 | Retain: 38.03 → 34.09
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank8_omega15_seed42

===== GD | Sine-LoRA | rank=16 | omega=15 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.4202 retain=3.6355
GD(Sine) rank=16 | Forget: 11.78 → 19.78 | Retain: 38.03 → 32.71
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank16_omega15_seed42


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoModelForCausalLM

RANKS = [2, 4, 8, 16]
OMEGA = 18

for RANK in RANKS:
    print(f"\n===== GD | Sine-LoRA | rank={RANK} | omega={OMEGA} =====")

    model = make_sine_lora(RANK, omega=OMEGA)

    forget_iter = Iterator(
        forget_train_ids,
        forget_train_mask,
        seed=SEED
    )
    retain_iter = Iterator(
        retain_train_ids,
        retain_train_mask,
        seed=SEED + 1
    )

    run_GD(
        model,
        forget_iter,
        retain_iter,
        lr=1e-5,
        steps=150,
        lambda_retain=5.0,
        clip=0.3
    )

    merged = model
    pF = ppl(merged, forget_eval_ids)
    pR = ppl(merged, retain_eval_ids)

    print(
        f"GD(Sine) rank={RANK} | "
        f"Forget: {BASE_FORGET:.2f} → {pF:.2f} | "
        f"Retain: {BASE_RETAIN:.2f} → {pR:.2f}"
    )

    save_dir = (
        f"{SAVE_ROOT}/GD_Sine_rank{RANK}"
        f"_omega{OMEGA}_seed{SEED}"
    )
    merged.save_pretrained(save_dir)
    print("saved:", save_dir)



===== GD | Sine-LoRA | rank=2 | omega=18 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2634 retain=3.7599
GD(Sine) rank=2 | Forget: 11.78 → 12.04 | Retain: 38.03 → 36.16
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank2_omega18_seed42

===== GD | Sine-LoRA | rank=4 | omega=18 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2941 retain=3.7244
GD(Sine) rank=4 | Forget: 11.78 → 12.66 | Retain: 38.03 → 35.00
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank4_omega18_seed42

===== GD | Sine-LoRA | rank=8 | omega=18 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.3463 retain=3.6815
GD(Sine) rank=8 | Forget: 11.78 → 14.92 | Retain: 38.03 → 33.78
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank8_omega18_seed42

===== GD | Sine-LoRA | rank=16 | omega=18 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.4671 retain=3.6238
GD(Sine) rank=16 | Forget: 11.78 → 22.17 | Retain: 38.03 → 32.39
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank16_omega18_seed42


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoModelForCausalLM

RANKS = [2, 4, 8, 16]
OMEGA = 20

for RANK in RANKS:
    print(f"\n===== GD | Sine-LoRA | rank={RANK} | omega={OMEGA} =====")

    model = make_sine_lora(RANK, omega=OMEGA)

    forget_iter = Iterator(
        forget_train_ids,
        forget_train_mask,
        seed=SEED
    )
    retain_iter = Iterator(
        retain_train_ids,
        retain_train_mask,
        seed=SEED + 1
    )

    run_GD(
        model,
        forget_iter,
        retain_iter,
        lr=1e-5,
        steps=150,
        lambda_retain=5.0,
        clip=0.3
    )

    merged = model
    pF = ppl(merged, forget_eval_ids)
    pR = ppl(merged, retain_eval_ids)

    print(
        f"GD(Sine) rank={RANK} | "
        f"Forget: {BASE_FORGET:.2f} → {pF:.2f} | "
        f"Retain: {BASE_RETAIN:.2f} → {pR:.2f}"
    )

    save_dir = (
        f"{SAVE_ROOT}/GD_Sine_rank{RANK}"
        f"_omega{OMEGA}_seed{SEED}"
    )
    merged.save_pretrained(save_dir)
    print("saved:", save_dir)



===== GD | Sine-LoRA | rank=2 | omega=20 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2672 retain=3.7562
GD(Sine) rank=2 | Forget: 11.78 → 12.14 | Retain: 38.03 → 36.01
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank2_omega20_seed42

===== GD | Sine-LoRA | rank=4 | omega=20 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.3001 retain=3.7155
GD(Sine) rank=4 | Forget: 11.78 → 12.86 | Retain: 38.03 → 34.81
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank4_omega20_seed42

===== GD | Sine-LoRA | rank=8 | omega=20 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.3439 retain=3.6723
GD(Sine) rank=8 | Forget: 11.78 → 14.43 | Retain: 38.03 → 33.55
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank8_omega20_seed42

===== GD | Sine-LoRA | rank=16 | omega=20 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.4691 retain=3.6135
GD(Sine) rank=16 | Forget: 11.78 → 28.55 | Retain: 38.03 → 32.27
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank16_omega20_seed42


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoModelForCausalLM

RANKS = [2, 4, 8, 16]
OMEGA = 22

for RANK in RANKS:
    print(f"\n===== GD | Sine-LoRA | rank={RANK} | omega={OMEGA} =====")

    model = make_sine_lora(RANK, omega=OMEGA)

    forget_iter = Iterator(
        forget_train_ids,
        forget_train_mask,
        seed=SEED
    )
    retain_iter = Iterator(
        retain_train_ids,
        retain_train_mask,
        seed=SEED + 1
    )

    run_GD(
        model,
        forget_iter,
        retain_iter,
        lr=1e-5,
        steps=150,
        lambda_retain=5.0,
        clip=0.3
    )

    merged = model
    pF = ppl(merged, forget_eval_ids)
    pR = ppl(merged, retain_eval_ids)

    print(
        f"GD(Sine) rank={RANK} | "
        f"Forget: {BASE_FORGET:.2f} → {pF:.2f} | "
        f"Retain: {BASE_RETAIN:.2f} → {pR:.2f}"
    )

    save_dir = (
        f"{SAVE_ROOT}/GD_Sine_rank{RANK}"
        f"_omega{OMEGA}_seed{SEED}"
    )
    merged.save_pretrained(save_dir)
    print("saved:", save_dir)



===== GD | Sine-LoRA | rank=2 | omega=22 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2688 retain=3.7540
GD(Sine) rank=2 | Forget: 11.78 → 12.18 | Retain: 38.03 → 35.92
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank2_omega22_seed42

===== GD | Sine-LoRA | rank=4 | omega=22 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2981 retain=3.7172
GD(Sine) rank=4 | Forget: 11.78 → 12.90 | Retain: 38.03 → 34.68
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank4_omega22_seed42

===== GD | Sine-LoRA | rank=8 | omega=22 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.3533 retain=3.6677
GD(Sine) rank=8 | Forget: 11.78 → 15.02 | Retain: 38.03 → 33.39
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank8_omega22_seed42

===== GD | Sine-LoRA | rank=16 | omega=22 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.4945 retain=3.6097
GD(Sine) rank=16 | Forget: 11.78 → 66.73 | Retain: 38.03 → 32.28
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank16_omega22_seed42


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoModelForCausalLM

RANKS = [2, 4, 8, 16]
OMEGA = 25

for RANK in RANKS:
    print(f"\n===== GD | Sine-LoRA | rank={RANK} | omega={OMEGA} =====")

    model = make_sine_lora(RANK, omega=OMEGA)

    forget_iter = Iterator(
        forget_train_ids,
        forget_train_mask,
        seed=SEED
    )
    retain_iter = Iterator(
        retain_train_ids,
        retain_train_mask,
        seed=SEED + 1
    )

    run_GD(
        model,
        forget_iter,
        retain_iter,
        lr=1e-5,
        steps=150,
        lambda_retain=5.0,
        clip=0.3
    )

    merged = model
    pF = ppl(merged, forget_eval_ids)
    pR = ppl(merged, retain_eval_ids)

    print(
        f"GD(Sine) rank={RANK} | "
        f"Forget: {BASE_FORGET:.2f} → {pF:.2f} | "
        f"Retain: {BASE_RETAIN:.2f} → {pR:.2f}"
    )

    save_dir = (
        f"{SAVE_ROOT}/GD_Sine_rank{RANK}"
        f"_omega{OMEGA}_seed{SEED}"
    )
    merged.save_pretrained(save_dir)
    print("saved:", save_dir)



===== GD | Sine-LoRA | rank=2 | omega=25 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2781 retain=3.7416
GD(Sine) rank=2 | Forget: 11.78 → 12.29 | Retain: 38.03 → 35.64
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank2_omega25_seed42

===== GD | Sine-LoRA | rank=4 | omega=25 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2978 retain=3.7103
GD(Sine) rank=4 | Forget: 11.78 → 12.92 | Retain: 38.03 → 34.50
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank4_omega25_seed42

===== GD | Sine-LoRA | rank=8 | omega=25 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.3630 retain=3.6565
GD(Sine) rank=8 | Forget: 11.78 → 15.18 | Retain: 38.03 → 33.12
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank8_omega25_seed42

===== GD | Sine-LoRA | rank=16 | omega=25 =====


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.5841 retain=3.6007
GD(Sine) rank=16 | Forget: 11.78 → 138.87 | Retain: 38.03 → 32.29
saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_Sine_rank16_omega25_seed42


# **GD on Lora**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoModelForCausalLM

In [ ]:
RANK = 2
model = make_lora(RANK)

forget_iter = Iterator(forget_train_ids, forget_train_mask, seed=SEED)
retain_iter = Iterator(retain_train_ids, retain_train_mask, seed=SEED+1)
run_GD(model, forget_iter, retain_iter, lr=1e-5, steps=150, lambda_retain=5, clip=0.3)

merged = model.merge_and_unload()
pF = ppl(merged, forget_eval_ids)
pR = ppl(merged, retain_eval_ids)

print("GD  Forget:", BASE_FORGET, "->", pF, " | Retain:", BASE_RETAIN, "->", pR)

save_dir = f"{SAVE_ROOT}/GD_rank{RANK}_seed{SEED}_lr1e-5_steps150_lambda5"
merged.save_pretrained(save_dir)
print("saved:", save_dir)


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

trainable params: 110,592 || all params: 125,309,184 || trainable%: 0.0883
[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2502 retain=3.8049
GD  Forget: 11.77557685978259 -> 11.791145458586213  | Retain: 38.02937180100788 -> 37.92062816250141


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_rank2_seed42_lr1e-5_steps150_lambda5


In [ ]:
RANK = 4
model = make_lora(RANK)

forget_iter = Iterator(forget_train_ids, forget_train_mask, seed=SEED)
retain_iter = Iterator(retain_train_ids, retain_train_mask, seed=SEED+1)
run_GD(model, forget_iter, retain_iter, lr=1e-5, steps=150, lambda_retain=5, clip=0.3)

merged = model.merge_and_unload()
pF = ppl(merged, forget_eval_ids)
pR = ppl(merged, retain_eval_ids)

print("GD  Forget:", BASE_FORGET, "->", pF, " | Retain:", BASE_RETAIN, "->", pR)

save_dir = f"{SAVE_ROOT}/GD_rank{RANK}_seed{SEED}_lr1e-5_steps150_lambda5"
merged.save_pretrained(save_dir)
print("saved:", save_dir)


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

trainable params: 221,184 || all params: 125,419,776 || trainable%: 0.1764
[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2525 retain=3.8012
GD  Forget: 11.77557685978259 -> 11.820139898533313  | Retain: 38.02937180100788 -> 37.807927538624895


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_rank4_seed42_lr1e-5_steps150_lambda5


In [ ]:
RANK = 8
model = make_lora(RANK)

forget_iter = Iterator(forget_train_ids, forget_train_mask, seed=SEED)
retain_iter = Iterator(retain_train_ids, retain_train_mask, seed=SEED+1)
run_GD(model, forget_iter, retain_iter, lr=1e-5, steps=150, lambda_retain=5, clip=0.3)

merged = model.merge_and_unload()
pF = ppl(merged, forget_eval_ids)
pR = ppl(merged, retain_eval_ids)

print("GD  Forget:", BASE_FORGET, "->", pF, " | Retain:", BASE_RETAIN, "->", pR)

save_dir = f"{SAVE_ROOT}/GD_rank{RANK}_seed{SEED}_lr1e-5_steps150_lambda5"
merged.save_pretrained(save_dir)
print("saved:", save_dir)


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

trainable params: 442,368 || all params: 125,640,960 || trainable%: 0.3521
[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2544 retain=3.7964
GD  Forget: 11.77557685978259 -> 11.850843510469387  | Retain: 38.02937180100788 -> 37.64644205821168


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_rank8_seed42_lr1e-5_steps150_lambda5


In [ ]:
RANK = 16
model = make_lora(RANK)

forget_iter = Iterator(forget_train_ids, forget_train_mask, seed=SEED)
retain_iter = Iterator(retain_train_ids, retain_train_mask, seed=SEED+1)
run_GD(model, forget_iter, retain_iter, lr=1e-5, steps=150, lambda_retain=5, clip=0.3)

merged = model.merge_and_unload()
pF = ppl(merged, forget_eval_ids)
pR = ppl(merged, retain_eval_ids)

print("GD  Forget:", BASE_FORGET, "->", pF, " | Retain:", BASE_RETAIN, "->", pR)

save_dir = f"{SAVE_ROOT}/GD_rank{RANK}_seed{SEED}_lr1e-5_steps150_lambda5"
merged.save_pretrained(save_dir)
print("saved:", save_dir)


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

trainable params: 884,736 || all params: 126,083,328 || trainable%: 0.7017
[GD] 0/150 forget=2.1389 retain=4.0061
[GD] 100/150 forget=2.2606 retain=3.7844
GD  Forget: 11.77557685978259 -> 11.9499465174378  | Retain: 38.02937180100788 -> 37.31761680970283


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved: /content/drive/My Drive/Lora Exp/Unlearned_models/LoraGDnLayers/new_version/GD_rank16_seed42_lr1e-5_steps150_lambda5
